# 01 — Explore the imported Fruit v1 dataset

Run the ZIP importer first. Preserve the supplied train/valid/test split. Inspect the audit, class imbalance, and label-quality concerns before training.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'configs/dataset.yaml').is_file():
    raise RuntimeError('Open this notebook from the FruitVision root or notebooks directory.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [ ]:
import json
audit_path = ROOT / 'outputs/dataset_audit_baseline.json'
if not audit_path.is_file():
    raise FileNotFoundError('Run python src/audit_dataset.py after importing the dataset.')
audit = json.loads(audit_path.read_text())
print(audit['objects_per_class'])
print('Current objects:', audit['total_objects'])
print('Empty labels:', audit['empty_label_files'])
print('Exact duplicate images:', audit['exact_duplicate_images'])


In [ ]:
from src.preprocessing import validate_dataset, inspect_pairs
config, report = validate_dataset(ROOT / 'configs/dataset.yaml')
report


In [ ]:
from collections import Counter
from src.visualization import plot_counts
plot_counts(report['train']['objects_per_class'], ROOT / 'outputs/figures/train_class_counts.png', title='Training class distribution', ylabel='Annotated objects')
from IPython.display import display
from PIL import Image
display(Image.open(ROOT / 'outputs/figures/train_class_counts.png'))
records = inspect_pairs(Path(config['path']) / 'images/train', Path(config['path']) / 'labels/train')
Counter((record['width'], record['height']) for record in records).most_common(10)


## Inspect a ground-truth image

These rectangles are dataset annotations, not predictions. Change the index to inspect more examples. Do the boxes enclose the correct fruit?

In [ ]:
from PIL import ImageDraw
record = records[0]
with Image.open(record['image']) as source:
    preview = source.convert('RGB')
draw = ImageDraw.Draw(preview)
W, H = preview.size
for class_id, x, y, width, height in record['boxes']:
    corners = ((x-width/2)*W, (y-height/2)*H, (x+width/2)*W, (y+height/2)*H)
    draw.rectangle(corners, outline='red', width=3)
    draw.text((corners[0], corners[1]), config['names'][class_id], fill='red')
display(preview)


## Record your observations

- What is the source and license?
- Are all ten classes represented in each split?
- Are objects small, overlapping, or partially hidden?
- Could related scenes leak between splits?
- Which labels need correction?